# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

I will use a Random Forest model for the Content Refresh Prioritization lane.

The goal is to rank content items for editorial review using several historical search and content signals. A Random Forest can combine nonlinear relationships between signals such as impressions, CTR, search position, content age, and time since the last update.

This is more flexible than the Week-4 hand-written baseline, which mainly uses staleness and visibility. The purpose is not to use a more complex model for its own sake, but to test whether combining multiple pre-decision signals improves the quality of the highest-priority recommendations.

I will evaluate the model using Precision@K and compare it with the Week-4 baseline on the same evaluation data and split.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 5.17 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/flyrank-ml-internship-starter
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

I will use a client-grouped train/test split so that pages from the same client do not appear in both training and test data.

This is more honest than randomly splitting individual pages because pages belonging to the same client can share characteristics. Holding out complete clients tests whether the model can generalize to clients it did not see during training.

I will use the same held-out groups when comparing the Random Forest with the Week-4 baseline, so the comparison uses the same evaluation population.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

print(
    "Client overlap:",
    len(set(train["client_id"]) & set(test["client_id"]))
)

Train rows: 22885
Test rows: 7115
Train clients: 24
Test clients: 8
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Target and Evaluation

For evaluation, I will use the observed declining outcome represented by `trend_direction == "down"`.

This outcome is used only as the evaluation target. The model features will exclude `trend_direction`, `trend_pct`, and `is_declining_label` because these fields are derived from the outcome and would leak the answer into the model.

The Random Forest and the Week-4 baseline will be evaluated on the same held-out clients using Precision@K.

In [5]:
# 1. Define the observed evaluation target
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down")
).astype(int)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# 2. Features available before the decision
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "search_volume",
    "competition",
    "word_count"
]

X_train = train[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

X_test = test[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y_train = train["is_declining_label"]
y_test = test["is_declining_label"]

# 3. Train the Random Forest
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 4. Model scores
model_score = model.predict_proba(X_test)[:, 1]

# 5. Week-4 baseline score
test["baseline_score"] = (
    (test["days_since_last_update"] >= 180).astype(int)
    *
    (test["impressions_90d"] >= 500).astype(int)
    *
    test["impressions_90d"]
)

baseline_score = test["baseline_score"].values

# 6. Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()

# 7. Compare
results = []

for k in [10, 20, 50]:
    baseline_precision = precision_at_k(
        baseline_score, y_test, k
    )

    model_precision = precision_at_k(
        model_score, y_test, k
    )

    results.append({
        "K": k,
        "Baseline_Precision": baseline_precision,
        "Model_Precision": model_precision,
        "Difference": model_precision - baseline_precision
    })

results_df = pd.DataFrame(results)

print("Model vs Baseline")
results_df# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Model vs Baseline


,K,Baseline_Precision,Model_Precision,Difference
0,10,0.60,0.40,-0.2
1,20,0.50,0.40,-0.1
2,50,0.62,0.52,-0.1


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

I will inspect which features the model relies on most and where its recommendations differ from the simple baseline.

Feature importance will be interpreted as model association, not causation. A highly important feature does not mean that changing that feature will cause the page's search performance to improve.

I will also inspect highly ranked model recommendations and disagreement cases. These cases help identify where the model finds patterns that the simple baseline does not capture, as well as cases where the model may make weak recommendations.

The results will be treated as observed and directional decision-support evidence rather than proof that refreshing a page will cause better search performance.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Top model features:")
display(importance.head(10))

# Inspect top model-ranked pages
error_review = test[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "trend_direction"
    ]
].copy()

error_review["model_score"] = model_score
error_review["baseline_score"] = baseline_score

error_review = error_review.sort_values(
    "model_score",
    ascending=False
)

display(error_review.head(10))

# Pages where the model gives a high score but the baseline gives zero
disagreement = error_review[
    (error_review["model_score"] > error_review["model_score"].median()) &
    (error_review["baseline_score"] == 0)
]

print("Model-high / baseline-zero cases:", len(disagreement))

display(disagreement.head(10))

Top model features:


,feature,importance
2,impressions_90d,0.156896
8,avg_position,0.149899
0,content_age_days,0.112183
13,word_count,0.093549
7,ctr,0.062447
10,scroll_rate,0.061412
4,pageviews_90d,0.059874
5,sessions_90d,0.054433
6,users_90d,0.054360
3,clicks_90d,0.046413


,content_id,client_id,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction,model_score,baseline_score
22922,content_f4d5f9725642,client_d029fa3a95,20,582,8.5,0.00,down,0.990,0
19155,content_ed35db35ef40,client_d029fa3a95,20,203,17.4,0.00,down,0.980,0
21882,content_2dbab51b83c9,client_d029fa3a95,20,781,9.1,0.00,up,0.975,0
18878,content_3ba5308bc87c,client_d029fa3a95,20,266,9.7,0.00,stable,0.970,0
28975,content_57aca9f1a5fe,client_d029fa3a95,20,1587,9.6,0.06,stable,0.970,0
17451,content_d015eb800625,client_d029fa3a95,20,1064,8.5,0.00,stable,0.970,0
21714,content_3324bf432188,client_d029fa3a95,20,594,8.5,0.00,stable,0.970,0
15539,content_0ee8d993ea59,client_d029fa3a95,20,513,12.1,0.00,up,0.965,0
18561,content_7a69792f4c29,client_d029fa3a95,20,257,10.3,0.00,down,0.965,0
18658,content_2ae8fb1778c0,client_d029fa3a95,20,391,11.4,0.00,down,0.960,0


Model-high / baseline-zero cases: 3523


,content_id,client_id,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction,model_score,baseline_score
22922,content_f4d5f9725642,client_d029fa3a95,20,582,8.5,0.00,down,0.990,0
19155,content_ed35db35ef40,client_d029fa3a95,20,203,17.4,0.00,down,0.980,0
21882,content_2dbab51b83c9,client_d029fa3a95,20,781,9.1,0.00,up,0.975,0
18878,content_3ba5308bc87c,client_d029fa3a95,20,266,9.7,0.00,stable,0.970,0
28975,content_57aca9f1a5fe,client_d029fa3a95,20,1587,9.6,0.06,stable,0.970,0
17451,content_d015eb800625,client_d029fa3a95,20,1064,8.5,0.00,stable,0.970,0
21714,content_3324bf432188,client_d029fa3a95,20,594,8.5,0.00,stable,0.970,0
15539,content_0ee8d993ea59,client_d029fa3a95,20,513,12.1,0.00,up,0.965,0
18561,content_7a69792f4c29,client_d029fa3a95,20,257,10.3,0.00,down,0.965,0
18658,content_2ae8fb1778c0,client_d029fa3a95,20,391,11.4,0.00,down,0.960,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.